# Лабораторная работа 3

## Часть 1: изюм

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.metrics import classification_report
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay, calibration_curve

**1. Предобработка**

Датасет с изюмом содержит характеристики двух видов изюма (измеренные по их изображениям, в пикселях; можно сказать, вы занимаетесь классификацией изображений!). Порисуйте графики, чтобы составить представление о распределении признаков *внутри класса*. Мы собираемся использовать наивный байесовский классификатор, LDA и QDА. Примените к признакам преобразования, изменяющие распределение там, где посчитаете нужным. Можно ли сразу по вашим графикам сказать, что предположения, нужные для какого-то из классификаторов, здесь совсем не выполняются?

- *чтобы убрать длинный хвост у распределения справа, можно взять корень или логарифм - это мы знаем. Если же длинный хвост у распределения слева, его можно отразить, прежде чем брать логарифмы.*

**2. Модели без калибровки**

Отделив 20% данных в тестовое множество, обучите NB, LDA и QDA. Вычислите accuracy, precision, recall, f1-score на тестовом множестве (можно сделать это с помощью `classification_report`, который печатает сразу много метрик)

Используя встроенные в эти классификаторы методы, вычислите также предсказываемые ими вероятности одного из классов, который мы обзовем положительным. Постройте кривые калибровки. Сделать это можно с помощью функции ниже, которую я украла из склерновского туториала и немного переделала, либо написав свою, более эстетичную версию. Будьте готовы прокомментировать увиденное и объяснить, что это за точки и как определяется, где их рисовать

In [ ]:
def draw_3_calibrations(clf_list, y_test, n_bins):
    """
    clf_list - список из трех tuples вида (модель, 'имя', вектор предсказанных на тесте вероятностей)
    y_test - реальные метки теста
    n_bins - сколько точек в кривой
    """
    fig = plt.figure(figsize=(10, 10))
    gs = GridSpec(3, 3)
    colors = plt.get_cmap("Dark2")

    ax_calibration_curve = fig.add_subplot(gs[:2, :3])
    calibration_displays = {}
    for i, (clf, name, y_prob) in enumerate(clf_list):
        disp = CalibrationDisplay.from_predictions(y_test, y_prob, name=name, n_bins=n_bins, ax=ax_calibration_curve, color=colors(i))
        calibration_displays[name] = disp

    ax_calibration_curve.grid()
    ax_calibration_curve.set_title("Calibration plots")

    grid_positions = [(2, 0), (2, 1), (2, 2)]
    for i, (_, name, _) in enumerate(clf_list):
        row, col = grid_positions[i]
        ax = fig.add_subplot(gs[row, col])

        ax.hist(
            calibration_displays[name].y_prob,
            range=(0, 1),
            bins=10,
            label=name,
            color=colors(i),
        )
        ax.set(title=name, xlabel="Predicted probability", ylabel="Count")

    plt.tight_layout()
    plt.show()

**3. Перекалибровка**

Для склерновских моделей перекалибровку вероятностей можно осуществить с помощью встроенного `CalibratedClassifierCV`. Сходите в справку и почитайте, что означает это CV и какие у него последствия. Для каждого базового классификатора обучите откалиброванный с методами `sigmoid` и `isotonic`, установив `ensemble=False`. Снова пересчитайте метрики и отрисуйте кривые калибровки (чтобы не громоздить их 9 штук на один рисунок, можете сделать 3 отдельных, по одному для каждой базовой модели)

Какой метод лучше откалибровал вероятности, если верить картинкам? Как калибровка повлияла на метрики - какому классификатору она была полезна, какому - не очень?

## Часть 2: дисбаланс классов

"На самом деле таска угадывания оттока\мошеннических операций по картам\кто не отдаст долг - это максимально типичная задача которая решается в жизни на каждом шагу половиной компаний у себя - поэтому нам надо это тоже посмотреть чтобы хотя бы примерно прикоснуться к тому с чем есть шанс работать в будущем" - сказал по этому поводу ваш прошлый преподаватель

**1. Предобработка** 

Загрузите данные (источник https://www.kaggle.com/datasets/blastchar/telco-customer-churn). Проверьте на наличие пропусков, определите, какие признаки категориальные, а какие - числовые. Предсказывать мы собираемся Churn

In [3]:
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.metrics import fbeta_score, make_scorer
from sklearn.model_selection import GridSearchCV

In [4]:
# если вы увидели расширение xls и попробовали прочитать это как эксель, возможно, у вас не получилось
data=pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.xls')
data.iloc[0]

customerID                7590-VHVEG
gender                        Female
SeniorCitizen                      0
Partner                          Yes
Dependents                        No
tenure                             1
PhoneService                      No
MultipleLines       No phone service
InternetService                  DSL
OnlineSecurity                    No
OnlineBackup                     Yes
DeviceProtection                  No
TechSupport                       No
StreamingTV                       No
StreamingMovies                   No
Contract              Month-to-month
PaperlessBilling                 Yes
PaymentMethod       Electronic check
MonthlyCharges                 29.85
TotalCharges                   29.85
Churn                             No
Name: 0, dtype: object

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Табличка выше могла ввести нас в заблуждение. Обратите особое внимание на колонку TotalCharges. Осознав, что с ней не так, пересмотрите свои выводы по пропускам и категориальности остальных колонок.

Посмотрите на дисбаланс классов, на расположение пропусков, на другие потенциально связанные с этими пропусками колонки и решите, что вы хотите с ними делать

Сделав все, что хотели, поделите данные на трейн и тест и проверьте, что соотношения классов в них примерно одинаковы

In [ ]:
# если в процессе вы не избавились от датафрейма, можно сделать это за счет normalize
train, test = train_test_split(data, test_size=0.2, random_state=2)
train['Churn'].value_counts(normalize=True), test['Churn'].value_counts(normalize=True)

**2. Логистическая регрессия** 

Предобработайте признаки, чтобы их можно было подавать в регрессию. Будет ли колонка `TotalCharges` избыточной или наоборот, потенциально полезной? Обратите внимание, что при one-hot кодировании вы почти наверняка создаете лишнюю колонку. Если вы не ожидаете, что в тестовых данных появится какая-то неизвестная прежде категория, то значение в одной one-hot колонке линейно выражается через значения в остальных. В `get_dummies` или `OneHotEncoder`'e есть аргумент, позволяющий лишнюю колонку убрать!

Загрузите `LogisticRegression` из sklearn и обучите ее без учета неравенства классов. Вычислите метрики на тестовом и тренировочном наборах: accuracy, precision, recall, f1, roc-auc. Вычислите те же метрики, просто предсказав везде, что "оттока нет", и сделайте вывод о том, какие из них имеют ценность в данной задаче.

Постройте roc- и pr- кривые и объясните, что они такое, почему они так выглядят

- *для построения кривых у sklearn есть готовые функции, правда сами кривые они не рисуют, а только возвращают списки координат, которые надо будет подать в plot, и списки соответствующих границ. Что может быть даже полезно, потому что если бы они отдавали вам только картинки, то вы не могли бы потом понять, какой границе соответствует та замечательная точка графика, где достигается такой хороший баланс между precision и recall*

**3. Дисбаланс классов** 

Когда мы выводили формулы для логистической регрессии, мы использовали предоположение о равенстве классов ($P(y=0)=P(y=1)$). Допустим, на самом деле соотношение между классами $\frac{P(y=0)}{P(y=1)}=\alpha$, но мы это проигнорировали и вычислили $P(y=1|X)$ и log odds (это $z=\log \frac{P(y=1|X)}{P(y=0|X)}$, и оно говорит нам о том, где проходит разделяющая плоскость) по старой формуле. 

1) Выведите, в каком соотношении находятся эти значения вероятностей с "честными". Предложите, что изменить в обучении или инференсе, чтобы учесть дисбаланс классов и снова считать правильные вероятности. Запишите!

2) Спуститесь в глубину архивов исходного кода sklearn для логистической регрессии (попасть туда можно, нажав \[source\] на страничке справки). Там найдите древние свитки, на которых записано, как дисбаланс классов обрабатывается в `LogisticRegression`. Перепишите эти знания предков в ноутбук и попробуйте объяснить, почему было сделано именно так.

3) Выбравшись из архивов, переобучите регрессию уже с учетом дисбаланса классов и пересчитайте все те же метрики. Посмотрите на precision-recall кривую, оцените, следует ли вам сдвинуть threshold с 0.5 куда-то еще. На основании чего вы можете сделать такой вывод?

**Дальше будет задание где нам нужны все наши модели одновременно, так что не удаляйте их из памяти пока!**

- *Для перебора параметров вообще можно использовать `GridSearchCV` или `RandomizedSearchCV`. Однако он требует выбрать некоторую метрику для сравнения*
- *заметьте, что никто не запрещает вам поставить в `class_weight` какие угодно коэффициенты дисбаланса. Как знать, может, нужная вам метрика станет лучше, если вы соврете о степени дисбаланса?*
- *`f1_score` - выбор желающих усидеть на двух стульях. Но что если мы хотим усидеть на одном стуле немного больше, чем на втором: мы как бы подозреваем, что, например, recall, важнее, но все же не хотим упускать из виду precision? Существует `fbeta_score` - взвешенная версия*

In [ ]:
# как-то вот так это может работать, если вы хотите еспользовать ее в переборе параметров
fb = make_scorer(fbeta_score, beta=3)
grid = GridSearchCV(base_classifier, param_grid={...}, scoring=fb, cv=3)

**4. Обыкновенная регрессия**

Попробуйте решить задачу с помощью `RidgeClassifier` и сравните метрики с логистической регрессией. Может, не так плоха линейная регрессия на друх классах?

Нам в этой лабе понадобятся вероятности, а у `RidgeClassifier` их нет, разве что какие-то confidence scores. Чтобы не посылать вас обратно копаться в исходном коде, просто скажем, что наши вероятности будут считаться взятием сигмоиды от того, что вернула `decision_function`

**5. Наивный Байес** 

Хотелось бы опробовать наивный байесовский классификатор на этой задаче, раз уж для L/QDA она не очень подходит в силу кучи категорий, но вот беда: наши признаки и категориальные, и числовые. Казалось бы, в чем проблема, но похоже в sklearn до сих пор нет версии байеса, которая бы умела с таким работать. Ваши варианты:

- подумать, как можно собрать математически эквивалентную версию из `GaussianNB` и `CategoricalNB`
- написать свой NB с нуля
- установить вот эту имплементацию https://github.com/remykarem/mixed-naive-bayes/tree/master

Выбирая вариант, учтите, что дальше нам от этого классификатора потребуются не просто ответы, а вероятности, или их подобие. У sklearn классификаторы обычно имеют метод `predict_proba`. В mixed-naive-bayes он какой-то тоже есть.

Прежде чем приступать к обучению классификатора, можно проверить кое-что еще. Наши численные признаки. Если вы не выбрали писать свой собственный классификатор, то заметьте, что готовый ожидает, что численные признаки нормально распределены. Но так ли это? Что вы можете сделать? Стоит ли оно того?

*логистическая регрессия с сигмоидой конечно тоже была заинтересована в нормальном распределении, но нам все равно пришлось бы скормить ей кучу категориальных фичей, так что...*

Обучите байесовский классификатор, посчитайте метрики

- *Для кодирования категориальных признаков для байеса лучше подойдет не one-hot, а `OrdinalEncoder`. Сравните его также с `LabelEncoder`: в чем разница?*

**6. Калибровка вероятностей**

1) Для наших классификаторов (логистическая регрессия, линейная регрессия, байесовский) постройте кривые калибровки вероятностей и прокомментируйте. 

> Как выглядят кривые калибровки для очень неуверенного классификатора? Для очень уверенного, но дающего неправильные ответы?

2. Попробуйте поправить вероятности. Для этого вообще существует `CalibratedClassifierCV`. Ниже приведен пример использования - разберитесь, что это за аргументы.

> Каждый из трех классификаторов засуньте вместе с тренировочным множеством в калибровщик. Попробуйте как `sigmoid`, так и `isotonic`. Постройте scatter графики старых и новых вероятностей, опишите, что с ними стало после калибровки. Отрисуйе новые кривые калибровки вместе со старыми (опять же, можно сделать три графика, отдельный для каждого классификатора).

In [ ]:
base_lr = LogisticRegression(...)# тут внутрь можно прописать ваши параметры, которые вы хотите зафиксировать
sigmoid_lr = CalibratedClassifierCV(base_lr, cv=4, method='sigmoid', ensemble=False)

Если вы написали своего наивного байеса, я за него не отвечаю, а вот если воспользовались готовым, то сейчас столкнетесь с проблемой: он не совсем совместим с этой калибровочной оболочкой. В частности, оболочка не верит, что это вообще классификатор и это вызывает проблемы в процессе. Возможно, вы придумаете какое-то более элегантное решение. Если вы тоже не особо программист, то, полистав исходный код, могу предложить вам следующий костыль: выдать ему фальшивое удостоверение классификатора и парочку классов для нашего конкретного случая.

In [ ]:
class ConvincingMixedNB(MixedNB):
    def __init__(self, categorical_features=None, max_categories=None,
                 alpha=0.5, priors=None, var_smoothing=1e-9):
        MixedNB.__init__(self, categorical_features, max_categories, alpha, priors, var_smoothing)
        self.classes_ = np.array([0,1])
        self._estimator_type = 'classifier'

Пересчитайте метрики, сравните со старыми. Также перерисуйте precision-recall кривые и сравните со старыми. Увиденное объясните.

**7.** Здесь было задание про овер-андерсэмплинг, но возможно стоит сжалиться над вами и пропустить его. Просто знайте, что нито вам не запрещает попробовать на этом датасете всяческие сэмплинги и посмотреть, что будет!